# C1: spaCy vektori reči + aglomerativno klasterovanje (Guardian)
**Tip:** profesorkin primer zadatka. Reprezentacija: vektori reči iz `en_core_web_md`. Algoritam: Ward.

### Zadatak
U fajlu `guardian_news_subset.csv` su novinski tekstovi lista Guardian. Relevantne su kolone `webTitle` i `bodyContent`.
- Kreirati kolonu `articleText` spajanjem `webTitle` i `bodyContent`.
- Transformisati `articleText` u vektore primenom **vektora reči iz spaCy medium modela**.
- Izvršiti **aglomerativno hijerarhijsko klasterovanje**. Optimalni broj klastera odrediti na osnovu **dendrograma i siluet skora** i obrazložiti.
- Za svaki klaster odrediti **top 10 reči / izraza po TF-IDF**.
- Odrediti veličinu klastera i opisati svaki klaster na osnovu ključnih reči i uvida u naslove.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import pdist

import matplotlib.pyplot as plt
import seaborn as sb
from pypalettes import load_cmap

from pathlib import Path
from IPython.display import display

import spacy

RAND_STATE = 1
DATA_DIR = Path.cwd().parent / 'data'
MODELS_DIR = Path.cwd().parent / 'models'

### Šta je urađeno
- Učitani podaci, izbačeno 17 članaka bez teksta, provereni duplikati.
- Napravljena `articleText` (naslov + tačka + tekst).
- Dužina članaka: prosek oko 800 reči. Tekstovi skraćeni na prvih 400 reči (`articleText_reduced`) da bi spaCy radio brže; početak članka nosi temu.
- Tekst je spreman za pretvaranje u vektore.

In [ ]:
data = pd.read_csv(DATA_DIR / 'guardian_news_subset.csv')
data.info()

In [ ]:
data = data.dropna(subset=['bodyContent']).reset_index(drop=True)
print('Duplikati:', data.duplicated().sum())
data['articleText'] = data['webTitle'] + '. ' + data['bodyContent']

In [ ]:
data['article_len_words'] = data['articleText'].str.split().str.len()
print(data['article_len_words'].describe())

data['articleText_reduced'] = data['articleText'].str.split().str[:400].str.join(' ')
print(data['articleText_reduced'].str.split().str.len().describe())

---
# ▶ OVDE KREĆEŠ

Sve iznad je gotovo. Od ove ćelije pa nadalje kucaš sam: kod, komentare u markdown ćelijama i tumačenja.

**Tvoji koraci:**

1. `spacy.load('en_core_web_md', disable=['parser','ner'])` i `spacy_srediTekst`
2. Vektori: za svaki tekst očisti, pa `nlp_pipe(" ".join(reci)).vector`, sve u `np.array` (n x 300)
3. `ward_linkage = sch.linkage(pdist(embeddings, metric='cosine'), method='ward')` i dendrogram
4. Silueta za kandidate: `1 - cosine_similarity`, `np.clip`, `fcluster`, `silhouette_score(..., metric='precomputed')`
5. Izbor broja klastera uz obrazloženje, `data['clust']`, `value_counts`
6. `get_cluster_keywords` (TF-IDF po klasteru, `mean(axis=0)`, top 10) i dati grafik
7. Nasumični naslovi po klasteru i opis klastera

---
## Dati kod
Ovo dobijaš uz zadatak.

In [ ]:
# DATI KOD (dobija se na kolokvijumu): prikaz kljucnih reci po klasterima
# Funkciju get_cluster_keywords(cluster_id) pises sam; treba da vrati listu parova (rec, skor).
# Raskomentarisi kad stignes do ovog koraka.

# fig, ax_grid = plt.subplots(nrows=2, ncols=3, figsize=(16, 10), constrained_layout=True)
# axes = ax_grid.flatten()
# cmap = load_cmap("excel_Median")
#
# n_clust = data['clust'].nunique()
# for i in range(n_clust):
#     scored_kwds = get_cluster_keywords(i+1)
#     scored_kwds = sorted(scored_kwds, key=lambda item: item[1])
#     scores = [s for kw, s in scored_kwds]
#     keywords = [kw for kw, s in scored_kwds]
#     axes[i].barh(keywords, scores, color=cmap(i))
#     axes[i].set_title(f"Cluster {i+1}", color=cmap(i))
#     axes[i].grid(visible=True, axis='y', color='gray', alpha=0.35)
#
# for ax in axes[n_clust:]:   # brise prazna polja ako je mreza veca od broja klastera
#     ax.remove()
#
# fig.suptitle("Kljucne reci po klasterima")
# plt.show()

In [ ]:
# DATI KOD (primer): skraceni dendrogram, ako je pun nepregledan
# plt.figure(figsize=(12, 6))
# sch.dendrogram(ward_linkage, truncate_mode='lastp', p=30)
# plt.title('Dendrogram (Ward)')
# plt.ylabel('Distanca')
# plt.show()